In [1]:
import numpy as np
import torch
import cv2
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

In [2]:
from datasets import load_dataset

ds = load_dataset("phunc20/nj_biergarten_captcha")

c:\Users\egtve\OneDrive\Documents\Uni\8. semester\TAI\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def char_to_1_hot(char, sorted_chars):
    # Convert a character to a one-hot encoded vector
    index = sorted_chars.index(char)
    one_hot = np.zeros(len(sorted_chars), dtype=np.uint8)
    one_hot[index] = 1.0
    return one_hot

def one_hot(label, sorted_chars):
    return np.hstack([char_to_1_hot(char, sorted_chars) for char in label[0]])

def one_hot_to_char(x, sorted_chars):
    y = np.array(x)
    y = y.squeeze()
    assert len(y) == len(sorted_chars)
    idx = np.argmax(y)
    return(sorted_chars[idx])

def one_hot_to_label(x, sorted_chars, char_per_label):
    y = np.array(x)
    y = y.squeeze()
    label_list = []
    assert len(y) == (len(sorted_chars * char_per_label))
    for i in range(0, char_per_label):
        start = i * len(sorted_chars)
        end = start + len(sorted_chars)
        label_list.append(one_hot_to_char(y[start:end], sorted_chars))
    return "".join(label_list)


In [4]:
def extract_labels(dataset):
    # Extract the labels from the dataset
    val = np.array(dataset.split("_")[1:])
    return val
    #*ds["train"][0]["__key__"].split("_")[1:]
class CustomCaptchaDataset(Dataset):
    def __init__(self, transform=None, sorted_chars=None):
        self.transform = transform
        self.sorted_chars = sorted_chars

    def __len__(self):
        return len(ds["train"])

    def __getitem__(self, idx):

        label = one_hot(extract_labels(ds["train"][idx]["__key__"]), self.sorted_chars)
        image = np.array(ds["train"][idx]["jpg"])

        if self.transform:
            image = self.transform(image)

        return image, label
    
    def get_labels(self, range):
        label = extract_labels(ds["train"][range]["__key__"])
        return label

In [5]:
data_points = 476118
batch_size = 128
char_per_label = 6

sorted_chars = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

train_size = int(data_points * 0.75)
test_size = int(data_points - train_size)
print(f"Train size: {train_size}, Test size: {test_size}")

# Grayscale the data
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Resize((25, 70)),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=0.1)
])

dataset = CustomCaptchaDataset(transform=transform, sorted_chars=sorted_chars)

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders for batch processing\n",
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(dataset[0][0].shape)
print(dataset[0][1])

Train size: 357088, Test size: 119030
torch.Size([3, 25, 70])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]


In [6]:
import torchvision.models as models
import torch.nn as nn
from tqdm import tqdm

model = models.resnet50(pretrained=True)


model.fc = nn.Linear(model.fc.in_features, char_per_label * len(sorted_chars))
model = model.to("cuda" if torch.cuda.is_available() else "cpu")


# Test if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available")
else:
    print("CUDA is not available")


c:\Users\egtve\OneDrive\Documents\Uni\8. semester\TAI\Project\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\egtve\OneDrive\Documents\Uni\8. semester\TAI\Project\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


CUDA is available


In [8]:
n_epochs = 20
# Initialize TensorBoard
writer = SummaryWriter(log_dir="runs/results")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.BCEWithLogitsLoss()
best_val_accuracy = 0.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("Using GPU")
# Training loop

for epoch in range(n_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for i, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Train]")):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Calculate training accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        train_total += labels.size(0)
        for i in range(predicted.size(0)):
            if torch.equal(predicted[i], labels[i]):
                train_correct += 1
        
    
    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total
    
    # Log training metrics
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Accuracy/train", train_accuracy, epoch)
    
    print(f"Epoch [{epoch + 1}/{n_epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")

    # Evaluation phase
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Valid]"):
            images = images.to("cuda" if torch.cuda.is_available() else "cpu")
            labels = labels.to("cuda" if torch.cuda.is_available() else "cpu")

            outputs = model(images)
            
            # Calculate validation loss
            val_loss = criterion(outputs, labels.float())
            val_running_loss += val_loss.item()
            
            # Calculate validation accuracy
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            val_total += labels.size(0)
            for i in range(predicted.size(0)):
                if torch.equal(predicted[i], labels[i]):
                    val_correct += 1

        val_loss = val_running_loss / len(test_loader)
        val_accuracy = 100 * val_correct / val_total
        
        # Log validation metrics
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Accuracy/val", val_accuracy, epoch)
        
        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    
    # Check if this is the best model based on validation accuracy
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            # Save the model
            torch.save(model.state_dict(), "best_model.pth")
            print(f"Model saved! New best validation accuracy: {val_accuracy:.2f}%")

Using GPU


Epoch 1/20 [Train]:   0%|          | 12/2790 [00:03<13:16,  3.49it/s]


KeyboardInterrupt: 